# 04 · Asal grup, pemilihan 109 label, dan fold

**File:** `04_Grup_Split_dan_Fold.ipynb`

Tujuan: menelusuri kode pembentuk kohor asli dan membuktikan bahwa pembacaan data saat ini memakai baris dan grup yang sama.

**Urutan belajar:** 03_Dataset_dan_Fitur.ipynb → notebook ini → 05_Model_dan_Bobot.ipynb.

**Mode:** penelusuran implementasi dan pemeriksaan data di memori. Run All tidak melatih model, tidak mengunduh data, dan tidak menimpa hasil penelitian. Setiap notebook membaca inputnya sendiri; tidak memerlukan variabel dari notebook lain.

Kode ML hanya memiliki satu sumber di `KODE_ML/03_Kode/`. Sel tampilan source membaca fungsi yang benar-benar diimpor; output kode itu bukan implementasi kedua. Sel berikutnya memanggil fungsi tersebut. Data/model besar tetap di folder artefak proyek.

## Persiapan
Cari folder KODE_ML dari lokasi notebook. Import mengambil kode lokal dan menampilkan lokasi aktual supaya asal kode dapat diperiksa.

In [1]:
from pathlib import Path
import sys
sys.dont_write_bytecode = True
cari = [Path.cwd(), *Path.cwd().parents]
KODE = next((p if (p / "03_Kode/runtime.py").is_file() and p.name == "KODE_ML"
             else p / "KODE_ML" for p in cari
             if (p / "03_Kode/runtime.py").is_file() and p.name == "KODE_ML"
             or (p / "KODE_ML/03_Kode/runtime.py").is_file()), None)
if KODE is None:
    raise FileNotFoundError("Buka notebook dari folder proyek atau KODE_ML.")
sys.path.insert(0, str(KODE.parent))
sys.path.insert(0, str(KODE))
from research.runtime import ROOT, read, digest
from belajar import tampilkan_sumber, tampilkan_awal
import pandas as pd
import numpy as np
from IPython.display import display, Markdown
pd.set_option("display.max_colwidth", 85)
print("Kode aktif:", KODE)


Kode aktif: C:\Users\ACER\Documents\Marvel\Skripsi\Project Skripsi\Perfume-MultiLabel-Classifier\KODE_ML


## Bedakan pembentukan historis dengan pemakaian saat ini

Source di asal_kohor.py berisi tiga fungsi asli pembentuk kohor yang dipertahankan secara AST. Fungsi ini hanya dipakai audit notebook, tidak dipanggil runner training utama. Tidak ada split baru yang disimpan.

In [2]:
import asal_kohor as asal
from research import dataset, splits
from research.preprocessing import load_config, taxonomy_from_ifra, load_sources
tampilkan_awal(asal)

**Import dan konstanta aktual:** `KODE_ML/asal_kohor.py`. SHA-256: `c0b4fab50851629f320eacb5c41b6d41b592d9fecd703dde1da13788e0c066e5`.

```python
"""Audit asal kohor beku. Bukan pembangun dataset baru atau fitur model 1024.

Tiga fungsi historis dipertahankan persis secara AST dari source pembentuk kohor.
Berikan fingerprint 1024 tersimpan untuk merekonstruksi identitas grup historis.
Fitur model utama tetap dibuat oleh research.features: Morgan 2048 + 8.
"""
import numpy as np
import pandas as pd
from rdkit import Chem
from sklearn.model_selection import GroupShuffleSplit
from research.splits import POLICY, inner_folds, validate_grouped_split

```

## structure_feature_groups

Menghubungkan molekul yang struktur tanpa stereonya identik ATAU fingerprint historisnya identik. Union-find menyatukan hubungan yang berantai. Ini bukan clustering semua molekul yang mirip berdasarkan Tanimoto.

| Bagian | Fungsi |
|---|---|
| parent | Awalnya setiap baris punya kelompok sendiri. |
| find | Cari wakil kelompok; path compression mempercepat pencarian berikutnya. |
| union | Gabungkan dua kelompok dengan wakil indeks terkecil agar hasil deterministik. |
| structures | Kunci canonical SMILES tanpa stereo menghubungkan struktur yang ekuivalen menurut aturan tersebut. Isotop/muatan tetap diperhatikan. |
| fingerprints | Byte fingerprint yang identik menghubungkan kelompok kedua; ini bukan batas kemiripan Tanimoto. |
| roots dan registry | Beri ID kelompok berurutan; tidak membuang baris molekul. |


In [3]:
tampilkan_sumber(asal.structure_feature_groups)

**Source aktual:** `KODE_ML/asal_kohor.py`, baris 16–48. Kode di bawah dibaca dari objek yang diimpor. SHA-256: `c0b4fab50851629f320eacb5c41b6d41b592d9fecd703dde1da13788e0c066e5`.

```python
def structure_feature_groups(smiles, morgan):
    if morgan.ndim != 2 or len(smiles) != len(morgan):
        raise ValueError("Grouping input shape mismatch.")
    parent = list(range(len(smiles)))

    def find(i):
        while parent[i] != i:
            parent[i] = parent[parent[i]]
            i = parent[i]
        return i

    def union(a, b):
        a, b = find(a), find(b)
        parent[max(a, b)] = min(a, b)

    structures, fingerprints = {}, {}
    for i, s in enumerate(smiles):
        mol = Chem.MolFromSmiles(s)
        if mol is None or len(Chem.GetMolFrags(mol)) != 1:
            raise ValueError("Grouping requires valid single molecules.")
        Chem.RemoveStereochemistry(mol)
        # Preserve isotope/charge; only remove stereo information for the grouping key.
        key = Chem.MolToSmiles(mol, isomericSmiles=True)
        fp = np.ascontiguousarray(morgan[i]).tobytes()
        if key in structures:
            union(i, structures[key])
        if fp in fingerprints:
            union(i, fingerprints[fp])
        structures.setdefault(key, i)
        fingerprints.setdefault(fp, i)
    roots = [find(i) for i in range(len(smiles))]
    registry = {root: j for j, root in enumerate(sorted(set(roots)))}
    return np.asarray([registry[r] for r in roots], dtype=np.int64)

```

## outer_split

GroupShuffleSplit mengalokasikan 20% kelompok ke test dengan seed 42. Karena ukuran kelompok berbeda, jumlah molekul test tidak harus tepat 20%.

In [4]:
tampilkan_sumber(asal.outer_split)

**Source aktual:** `KODE_ML/asal_kohor.py`, baris 51–55. Kode di bawah dibaca dari objek yang diimpor. SHA-256: `c0b4fab50851629f320eacb5c41b6d41b592d9fecd703dde1da13788e0c066e5`.

```python
def outer_split(groups, test_size, seed):
    # Test allocation uses structure groups only, never label scores or test performance.
    cv = GroupShuffleSplit(n_splits=1, test_size=test_size, random_state=seed)
    a, b = next(cv.split(np.zeros(len(groups)), groups=groups))
    return np.sort(a), np.sort(b)

```

## encode_and_split

Pertama membentuk y seluruh taksonomi; lalu menentukan train/test dari grup; kemudian menghitung dukungan hanya pada train. Syarat >=30 positif dan >=2 negatif memilih target, lalu membangun fold per label. Tidak ada top-109 atau pemilihan menggunakan skor test.

| Urutan blok | Alasan |
|---|---|
| y untuk seluruh taxonomy | Belum ada daftar 109 target; seluruh istilah IFRA di-encode dulu. |
| groups, train, test | Pembagian luar memakai kelompok struktur, bukan skor/performa label. |
| positive, minimum, chosen | Hanya dukungan train yang menentukan label; test tidak ikut menentukan syarat minimum. |
| labels dan target | Ambil kolom terpilih dengan urutan taksonomi tetap. Semua baris molekul dipertahankan. |
| inner_folds per label | Kelompok tetap utuh dan setiap label mempunyai stratifikasi biner sendiri. |
| support, selection | Catat dukungan dan alasan setiap label dipilih/tidak. |
| validate + return | Validasi pembagian, lalu kembalikan di memori untuk dibandingkan; notebook tidak menyimpan split baru. |


In [5]:
tampilkan_sumber(asal.encode_and_split)

**Source aktual:** `KODE_ML/asal_kohor.py`, baris 58–89. Kode di bawah dibaca dari objek yang diimpor. SHA-256: `c0b4fab50851629f320eacb5c41b6d41b592d9fecd703dde1da13788e0c066e5`.

```python
def encode_and_split(records, taxonomy, config, *, morgan=None):
    y = np.asarray([[int(label in row) for label in taxonomy] for row in records.labels], dtype=np.uint8)
    if morgan is None:
        morgan, _ = extract_features(records, config)
    groups = structure_feature_groups(records.smiles.tolist(), morgan)
    train, test = outer_split(groups, config["split"]["test_size"], config["split"]["seed"])
    positive = y[train].sum(axis=0)
    minimum = config["labels"]["min_train_positive"]
    chosen = [j for j in range(len(taxonomy)) if positive[j] >= minimum and len(train)-positive[j] >= 2]
    if not chosen:
        raise ValueError("No labels have enough training examples.")
    labels, target = [taxonomy[j] for j in chosen], y[:, chosen]
    folds, support = {}, []
    for j, label in enumerate(labels):
        pos = int(target[train,j].sum()); neg = len(train)-pos
        try:
            folds[label] = inner_folds(train, target[:,j], groups,
                                       config["split"]["max_folds"], config["split"]["seed"])
        except ValueError as exc:
            raise ValueError(f"Label {label}: {exc}") from exc
        support.append({"label":label,"train_positive":pos,"train_negative":neg,
                        "folds":len(folds[label]),
                        "positive_groups":len(set(groups[train][target[train,j] == 1])),
                        "negative_groups":len(set(groups[train][target[train,j] == 0]))})
    split = {"train":train.tolist(),"test":test.tolist(),"folds":folds,"eligible_labels":labels,
             "seed":config["split"]["seed"],"group_policy":POLICY,"groups":groups.tolist(),
             "outer_method":"GroupShuffleSplit","inner_method":"StratifiedGroupKFold",
             "test_size_unit":"groups"}
    validate_grouped_split(split, groups)
    selection = [{"label":label,"train_positive":int(positive[j]),"selected":j in chosen}
                 for j,label in enumerate(taxonomy)]
    return target, labels, split, pd.DataFrame(support), pd.DataFrame(selection)

```

## inner_folds

Untuk satu label, StratifiedGroupKFold membagi training berdasarkan grup dan keseimbangan kelas. Maksimum lima fold; bila satu kelas tidak layak, fungsi dapat mengurangi fold tanpa melihat skor model. Fold tiap label bisa berbeda.

In [6]:
tampilkan_sumber(splits.inner_folds)

**Source aktual:** `KODE_ML/03_Kode/splits.py`, baris 6–18. Kode di bawah dibaca dari objek yang diimpor. SHA-256: `9386cb3804ad90592c9a3b9e11cbffcb28023754ab069e10fc82c4aa1e90d2f0`.

```python
def inner_folds(train, target, groups, max_folds, seed):
    train = np.asarray(train, dtype=int)
    truth, train_groups = target[train], groups[train]
    if set(np.unique(truth)) != {0, 1}:
        raise ValueError("Grouped CV requires both target classes.")
    upper = min(max_folds, len(set(train_groups[truth == 1])), len(set(train_groups[truth == 0])))
    # Lower k only for class feasibility, never based on model scores. No random-split fallback.
    for k in range(upper, 1, -1):
        cv = StratifiedGroupKFold(n_splits=k, shuffle=True, random_state=seed)
        candidates = list(cv.split(np.zeros(len(train)), truth, train_groups))
        if all(len(np.unique(truth[a])) == 2 and len(np.unique(truth[b])) == 2 for a,b in candidates):
            return [{"train":train[a].tolist(), "validation":train[b].tolist()} for a,b in candidates]
    raise ValueError("No feasible grouped CV with both classes; keep molecules and review the protocol.")

```

## validate_grouped_split

Mengecek indeks tidak ganda, train/test tidak tumpang tindih, grup tidak melintas, dan validation mencakup seluruh training satu kali per label.

In [7]:
tampilkan_sumber(splits.validate_grouped_split)

**Source aktual:** `KODE_ML/03_Kode/splits.py`, baris 21–52. Kode di bawah dibaca dari objek yang diimpor. SHA-256: `9386cb3804ad90592c9a3b9e11cbffcb28023754ab069e10fc82c4aa1e90d2f0`.

```python
def validate_grouped_split(split, groups):
    groups = np.asarray(groups)
    if split.get("group_policy") != POLICY or split.get("groups") != groups.tolist():
        raise ValueError("Missing or inconsistent structure/feature groups.")
    universe = set(range(len(groups)))
    def partition(a, b, expected):
        if len(a) != len(set(a)) or len(b) != len(set(b)):
            raise ValueError("Duplicate split indices.")
        a, b = set(a), set(b)
        if not a or not b or a & b or a | b != expected:
            raise ValueError("Invalid grouped partition coverage.")
        if set(groups[list(a)]) & set(groups[list(b)]):
            raise ValueError("Structure/feature group leakage across partitions.")
    partition(split["train"], split["test"], universe)
    training = set(split["train"])
    if set(split["folds"]) != set(split["eligible_labels"]):
        raise ValueError("Fold label registry mismatch.")
    fold_count = 0
    for label, folds in split["folds"].items():
        if len(folds) < 2:
            raise ValueError("At least two grouped folds required.")
        seen = []
        for fold in folds:
            partition(fold["train"], fold["validation"], training)
            seen.extend(fold["validation"])
            fold_count += 1
        if sorted(seen) != sorted(training):
            raise ValueError("Grouped validation must cover training rows once.")
    counts = pd.Series(groups).value_counts()
    return {"group_policy":POLICY, "group_count":int(len(counts)),
            "largest_group":int(counts.max()), "group_overlap_train_test":0,
            "group_overlap_cv":0, "folds_verified":fold_count}

```

## Perbedaan teknis pada fallback historis

Kode aktual di `KODE_ML/asal_kohor.py` berikut menggantikan fallback pembuat fitur lama. Ia sengaja menolak pemanggilan tanpa argumen morgan historis. Fungsi encode_and_split tetap sama; notebook memberikan morgan tersimpan secara eksplisit. Ini pembatasan audit, bukan perubahan algoritma grup atau fitur model.

```python
def extract_features(records, config):
    raise ValueError("Audit asal kohor wajib menerima morgan historis tersimpan; tidak membuat fitur lama.")
```

## Rekonstruksi pembentukan kohor di memori

Kita membaca fingerprint historis 1024 HANYA untuk memeriksa asal kelompok lama. Tidak menghitung fitur 1024 baru, tidak memakainya sebagai X model utama, dan tidak menulis split. Fungsi asli di atas dipanggil dengan nilai morgan historis eksplisit.

In [8]:
config = load_config()
tables,_ = load_sources(config)
taxonomy = taxonomy_from_ifra(tables['ifra_behavior'])
records = pd.read_csv(dataset.DATASET/'records.csv')
import json
for column in ('labels','sources'):
    records[column] = records[column].map(json.loads)
stored = read(dataset.DATASET/'splits.json')
historical_morgan = np.empty((len(records),1024),dtype=np.uint8)
for role in ('train','test'):
    with np.load(dataset.DATASET/f'{role}.npz',allow_pickle=False) as z:
        historical_morgan[z['row_index']] = z['morgan']
target, labels, rebuilt, supports, selection = asal.encode_and_split(records,taxonomy,config,morgan=historical_morgan)
assert rebuilt == stored
assert labels == read(dataset.DATASET/'dataset_manifest.json')['labels']
for role in ('train','test'):
    with np.load(dataset.DATASET/f'{role}.npz',allow_pickle=False) as z:
        assert np.array_equal(target[rebuilt[role]],z['truth'])
display({'taksonomi':len(taxonomy),'label_terpilih':len(labels),
 'train':len(rebuilt['train']),'test':len(rebuilt['test']),
 'split_grup_fold_target_identik':True})
display(splits.validate_grouped_split(rebuilt,np.asarray(rebuilt['groups'])))
display(selection.groupby('selected').agg(jumlah_label=('label','size'),min_positif=('train_positive','min'),max_positif=('train_positive','max')))

{'taksonomi': 184,
 'label_terpilih': 109,
 'train': 5353,
 'test': 1333,
 'split_grup_fold_target_identik': True}

{'group_policy': 'structure-or-morgan-components-v1',
 'group_count': 5363,
 'largest_group': 38,
 'group_overlap_train_test': 0,
 'group_overlap_cv': 0,
 'folds_verified': 545}

,jumlah_label,min_positif,max_positif
selected,,,
False,75,1,28
True,109,32,1987


## Contoh pembagian satu label

Input: split beku. Output: jumlah train/validation dan bukti grup terpisah pada setiap fold. Ini lima fold untuk satu label, bukan membagi lima sumber dataset menjadi lima fold.

In [9]:
label=labels[0]
groups=np.asarray(rebuilt['groups'])
display(pd.DataFrame([{'label':label,'fold':j+1,'latih':len(f['train']),
 'validasi':len(f['validation']),'grup_overlap':len(set(groups[f['train']]) & set(groups[f['validation']]))}
 for j,f in enumerate(rebuilt['folds'][label])]))
protocol=read(KODE/'03_Kode/config/experiment.json')
print('Seed repeated CV:',protocol['robustness']['seeds'])

,label,fold,latih,validasi,grup_overlap
0,acidic,1,4266,1087,0
1,acidic,2,4302,1051,0
2,acidic,3,4258,1095,0
3,acidic,4,4296,1057,0
4,acidic,5,4290,1063,0


Seed repeated CV: [7, 21, 42, 84, 100]


Pada repeated CV, seed 7 dipakai untuk satu putaran fold, lalu seed 21 untuk putaran berikutnya, dan seterusnya. Seed adalah kunci pengacakan; nilai 7 tidak berarti mengulang tujuh kali. Random state estimator tetap 42 di make_model.

## Batas kesimpulan
Kesamaan seluruh indeks/fold membuktikan audit memakai pembagian asli. Ini tidak membuktikan molekul train dan test bebas dari segala kemiripan kimia. Test juga pernah dilihat pada pengembangan sebelumnya; hasil tetap eksploratori.

Lanjut ke notebook 05 untuk model dan bobot kelas.